# 05 – Ethics, Bias & Fairness Analysis

**Goal:** Surface the limitations, biases and ethical considerations baked into the data and the deployed pipeline. This notebook is referenced from the project documentation and is required for the bonus Ethics section.

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import mean_absolute_error

from src import config

sns.set_style('whitegrid')

## 1. Data Bias – StockX Dataset

The StockX Data Contest covers 2017-2019 and is heavily biased toward Off-White and Yeezy. We measure both biases here.

In [ ]:
train_df = pd.read_parquet(config.PROCESSED_TRAIN)
val_df = pd.read_parquet(config.PROCESSED_VAL)
test_df = pd.read_parquet(config.PROCESSED_TEST)
df = pd.concat([train_df, val_df, test_df])
print(f'Combined rows: {len(df)}')

### 1a. Geographic Bias

In [ ]:
if 'buyer_region' in df.columns:
    region_share = df['buyer_region'].value_counts(normalize=True) * 100
    fig, ax = plt.subplots(figsize=(10, 4))
    region_share.head(10).plot(kind='bar', ax=ax, color='#3b82f6')
    ax.set_title('Buyer Region Share (top 10)')
    ax.set_ylabel('Share of Transactions (%)')
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()
    print('Top-3 share:', region_share.head(3).sum().round(1), '%')
else:
    print('buyer_region not available')

**Finding:** StockX transactions are USA-dominated. Predictions therefore reflect US resell dynamics; markets like Asia and EU may behave differently.

### 1b. Temporal Bias

In [ ]:
if 'order_date' in df.columns:
    df['year'] = pd.to_datetime(df['order_date'], errors='coerce').dt.year
    fig, ax = plt.subplots(figsize=(8, 4))
    df['year'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#00d4aa')
    ax.set_title('Transactions per Year')
    ax.set_xlabel('Year')
    plt.tight_layout()
    plt.show()

**Finding:** All data is from 2017-2019. The Adidas-Yeezy separation in 2022, the rise of New Balance, and post-2020 market cooling are **not** reflected. Predictions are best interpreted as *historical* fair-value estimates, not real-time market quotes.

### 1c. Brand Imbalance

In [ ]:
brand_share = df['brand'].value_counts(normalize=True) * 100
fig, ax = plt.subplots(figsize=(7, 4))
brand_share.plot(kind='bar', ax=ax, color='#f39c12')
ax.set_title('Brand Share in Training Data')
ax.set_ylabel('Share (%)')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()
print(brand_share)

**Finding:** Off-White and Yeezy together make up the bulk of transactions. Brands like New Balance, Converse, Vans are under-represented and will likely receive less reliable predictions.

## 2. ML Bias – Residuals by Brand

Does the price model systematically over- or under-predict for any brand?

In [ ]:
model_path = config.ML_MODEL_PATH
if model_path.exists():
    model = joblib.load(model_path)
    X_test = test_df[config.ML_FEATURE_COLS].values
    y_test = test_df[config.ML_TARGET_COL].values
    y_pred = model.predict(X_test)
    test_df = test_df.copy()
    test_df['residual'] = y_test - y_pred
    test_df['abs_err'] = test_df['residual'].abs()

    by_brand = test_df.groupby('brand').agg(
        n=('residual', 'size'),
        mean_residual=('residual', 'mean'),
        mae=('abs_err', 'mean'),
    ).sort_values('n', ascending=False)
    print(by_brand)

    fig, ax = plt.subplots(figsize=(9, 4))
    by_brand['mean_residual'].plot(kind='bar', ax=ax, color='#e94560')
    ax.axhline(0, color='#1a1a2e', linewidth=1)
    ax.set_title('Mean Residual by Brand (positive = under-predicted)')
    ax.set_ylabel('Mean Residual ($)')
    plt.tight_layout()
    plt.show()
else:
    print('ML model not trained yet – run 03_ml_training.ipynb first.')

**Reading the chart:**
- A positive mean residual means the model **under**-predicts (resell price was higher
  than the prediction).
- Off-White's wide residual range reflects its long-tailed grail pricing.
- Brands with under 100 test rows are noisy – flag the limitation in the app.

## 3. CV Bias – Per-Class Fairness

(Run **after** `02_cv_training.ipynb` saves the model and the per-class F1 table.)

Look for: classes systematically below average F1, and whether colour patterns correlate with classification difficulty. The same per-class accuracy bar chart in `02_cv_training.ipynb` doubles as the fairness signal.

**Qualitative bias check:**
- All-white sneakers (AF1, Yeezy 350 Triple White) are confused with each other.
- Black colorways are often confused across Air Jordan 1 sub-styles.
- Heavily logo-marked sneakers (Off-White) are easier to classify – their visual
  signature is distinctive.

## 4. Socioeconomic Considerations

The sneaker resell market reinforces wealth disparities: buyers with capital flip limited drops, often pricing primary fans out. A predictive tool can:

- **Inform** retail buyers about realistic resell expectations (fair-value transparency).
- **Accelerate speculation** if used by flippers as a screening tool.

**Mitigation we apply:**
- The app frames itself as *informational*, not investment advice.
- Confidence and data-source disclaimers are always visible.
- The advisor explicitly mentions limitations (data age, USA-bias, brand imbalance).

## 5. Transparency & Limitations

What the model **cannot** do:

- **No authentication.** It classifies sneakers visually – it does not detect fakes.
- **No real-time pricing.** Predictions reflect 2017-2019 market dynamics.
- **No size-rarity premium modelling.** Extreme sizes (US <7 or >13) often command
  higher resells in reality; the model averages this away.
- **No condition assessment.** Inputs are assumed to be deadstock; worn pairs are
  systematically over-valued.

## 6. Disclaimer in the App

Every prediction in `app/app.py` is paired with:

> *Predictions are based on historical StockX data (2017-2019). Not investment advice. > The tool does not authenticate sneakers or detect counterfeits.*

Confidence scores below 50% trigger an explicit warning that the visual classification may be unreliable.

## Summary

| Source | Bias | Mitigation |
|---|---|---|
| StockX 2017-2019 | USA-dominated, brand-skewed, time-bounded | Disclaimer in UI, ethics section in docs |
| Sneaker images | Class imbalance (Off-White/Yeezy oversampled) | Per-class F1 reporting, weighted scoring considered |
| Price model | Higher error on >$1000 grails and under $50 segment | Segment-level reporting in residual analysis |
| LLM advisor | Hallucination risk for rare models, confidence-blind verdicts | Fallback templates, prompt grounded in knowledge base |
| Use case | Risk of fuelling speculation | Tool framed as informational, no buy-button, explicit disclaimer |